In [1]:
import torch
from typing import Dict, List, Literal, Optional, Tuple

In [ ]:


CheckMode = Literal["all_points", "mean", "final_point"]
Metric = Literal["l_inf", "l1", "l2"]

def agent_traj_correct(
    pred: torch.Tensor,          # [B, A, H, 2]
    targets: torch.Tensor,       # [B, H, A, 7]  (we use only :2)
    targets_mask: torch.Tensor,  # [B, H, A]     (1 valid, 0 pad)
    threshold: float,
    mode: CheckMode = "all_points",
    metric: Metric = "l2",
    return_bool: bool = False,
    denorm: bool = False, # set True if inputs are normalized and need to be denormalized
    std_xy: Optional[torch.Tensor] = None,    # ideally [2]
    mean_xy: Optional[torch.Tensor] = None,   # ideally [2]
    verbose: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Returns:
      correct:    [B, A] (int {0,1} by default, or bool if return_bool=True)
      agent_valid:[B, A] (bool) True iff the agent has at least 1 valid future step
    """

    # --- align shapes ---
    if pred.ndim != 4 or pred.size(-1) != 2:
        raise ValueError(f"pred must be [B,A,H,2], got {tuple(pred.shape)}")
    if targets.ndim != 4 or targets.size(-1) < 2:
        raise ValueError(f"targets must be [B,H,A,7] (>=2), got {tuple(targets.shape)}")
    if targets_mask.ndim != 3:
        raise ValueError(f"targets_mask must be [B,H,A], got {tuple(targets_mask.shape)}")

    B, A, H, _ = pred.shape

    if denorm:
        pred_xy_denorm = pred[..., :2] * std_xy + mean_xy               # [B, A, H, 2]
        targets_xy_denorm = (targets[..., :2] * std_xy + mean_xy)       # [B, H, A, 2]
        pred = pred_xy_denorm
        targets[..., :2] = targets_xy_denorm[..., :2]

    # targets_xy: [B,H,A,2] -> [B,A,H,2]
    targets_xy = targets[..., :2].permute(0, 2, 1, 3).contiguous()
    # mask: [B,H,A] -> [B,A,H]
    m = targets_mask.permute(0, 2, 1).contiguous()
    m_bool = m.to(dtype=torch.bool)

    # agent is "valid" if it has any valid timestep
    agent_valid = m_bool.any(dim=-1)  # [B,A]

    # --- per-timestep distance ---
    diff = pred - targets_xy  # [B,A,H,2]

    if metric == "l_inf":
        per_point = diff.abs().amax(dim=-1)                # max(|dx|,|dy|)
    elif metric == "l1":
        per_point = diff.abs().sum(dim=-1)                 # |dx|+|dy|
    elif metric == "l2":
        per_point = torch.sqrt((diff * diff).sum(dim=-1))  # sqrt(dx^2+dy^2)
    else:
        raise ValueError(f"Unknown metric: {metric}")

    thr = torch.as_tensor(threshold, device=pred.device, dtype=per_point.dtype)

    # --- modes ---
    if mode == "all_points":
        # Option 1: all valid points must be <= threshold
        ok_t = (per_point <= thr) | (~m_bool)   # invalid timesteps don't count against you
        correct = ok_t.all(dim=-1) & agent_valid
        if verbose:
            print("All points correct:", correct)
    elif mode == "mean":
        # Option 2: mean over valid points must be <= threshold
        denom = m.to(per_point.dtype).sum(dim=-1).clamp_min(1.0)     # [B,A]
        mean_err = (per_point * m.to(per_point.dtype)).sum(dim=-1) / denom
        correct = (mean_err <= thr) & agent_valid
        if verbose:
            print("Denom:", denom, " Mean_err:", mean_err)
    elif mode == "final_point":
        # Option 3: last valid timestep must be <= threshold
        B, A, H = m_bool.shape
        t_idx = torch.arange(H, device=pred.device).view(1, 1, H)
        last_idx = (t_idx * m_bool.long()).amax(dim=-1)  # [B,A]
        last_err = per_point.gather(dim=-1, index=last_idx.unsqueeze(-1)).squeeze(-1)
        correct = (last_err <= thr) & agent_valid
        if verbose:
            print("Last idx:", last_idx, " Last_err:", last_err)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    if return_bool:
        return correct, agent_valid
    return correct.long(), agent_valid

In [ ]:
def batch_correct_by_agent_count(
    pred: torch.Tensor,          # [B, A, H, 2]
    targets: torch.Tensor,       # [B, H, A, 7]
    targets_mask: torch.Tensor,  # [B, H, A]
    threshold: float,
    mode: CheckMode = "all_points",
    metric: Metric = "l2",
    # If provided: output 1 iff at least N agents are correct.
    # BUT if T valid agents < N, then require all T to be correct (still output 1 if all valid are correct).
    n_correct_upper_bound: Optional[int] = None,
    return_counts: bool = True,
    denorm: bool = False, # set True if inputs are normalized and need to be denormalized
    std_xy: Optional[torch.Tensor] = None,    # ideally [2]
    mean_xy: Optional[torch.Tensor] = None,   # ideally [2]
    verbose: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, Optional[torch.Tensor]]:
    """
    Returns:
      sample_ok:     [B] int {0,1}  -- whether the sample passes the criterion
      num_correct:   [B] long       -- number of valid agents that are correct
      num_valid:     [B] long       -- number of valid agents
      correct_mask:  [B, A] long {0,1} or None  -- per-agent correctness (only if return_counts=True)

    Behavior:
      - Always counts correctness only over VALID agents (agent has >=1 valid timestep).
      - If n_correct_upper_bound is None:
          sample_ok = 1 iff ALL valid agents are correct (vacuously 1 if no valid agents).
      - If n_correct_upper_bound = N:
          Let T = #valid agents in the sample.
          Required correct agents = min(N, T).
          sample_ok = 1 iff num_correct >= min(N, T).
          (So when T < N, you effectively require "all valid agents are correct".)
    """
    correct_agent, agent_valid = agent_traj_correct(
        pred=pred,
        targets=targets,
        targets_mask=targets_mask,
        threshold=threshold,
        mode=mode,
        metric=metric,
        return_bool=False,
        denorm=denorm,
        std_xy=std_xy,
        mean_xy=mean_xy,
        verbose=verbose,
    )  # correct_agent: [B,A] {0,1}, agent_valid: [B,A] bool

    correct_agent = correct_agent.to(torch.long)
    agent_valid_l = agent_valid.to(torch.long)

    # Count only valid agents
    correct_valid = correct_agent * agent_valid_l  # [B,A]
    num_correct = correct_valid.sum(dim=1)         # [B]
    num_valid = agent_valid_l.sum(dim=1)           # [B]

    if n_correct_upper_bound is None:
        # all valid agents must be correct
        sample_ok = (num_correct == num_valid).to(torch.long)
    else:
        if n_correct_upper_bound < 0:
            raise ValueError("n_correct_upper_bound must be >= 0")
        required = torch.minimum(
            num_valid,
            torch.tensor(n_correct_upper_bound, device=num_valid.device, dtype=num_valid.dtype),
        )
        sample_ok = (num_correct >= required).to(torch.long)

    if return_counts:
        return sample_ok, num_correct, num_valid, correct_agent
    else:
        return sample_ok, num_correct, num_valid, None

In [4]:
GenMode = Literal["random_pred", "partial_corrupt"]

def generate_synthetic_batch(
    device: Optional[torch.device] = None,
    dtype: torch.dtype = torch.float32,
    seed: Optional[int] = None,
    # Fixed sizes (as you requested)
    B: int = 4,
    A: int = 8,
    H: int = 12,
    # Mask / trajectory generation controls
    p_agent_valid: float = 0.75,          # probability an agent exists (has >=1 valid step)
    valid_len_min: int = 1,
    valid_len_max: int = 12,              # <= H
    gt_start_scale: float = 10.0,         # initial position scale
    gt_step_scale: float = 0.5,           # random walk step scale
    # Pred generation mode
    mode: GenMode = "random_pred",
    # mode="random_pred"
    pred_random_scale: float = 10.0,
    # mode="partial_corrupt"
    n_corrupt_agents: int = 2,            # N valid agents to corrupt (per sample)
    seg_len_min: int = 2,                 # K range
    seg_len_max: int = 5,
    corrupt_random_scale: float = 10.0,   # how “random” corrupted segment is
    # Optional: if you'd rather corrupt by adding noise instead of fully random,
    # set corrupt_as_noise=True (uses gt + noise instead of fully random)
    corrupt_as_noise: bool = False,
    corrupt_noise_scale: float = 3.0,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, Dict]:
    """
    Generates:
      pred         [B, A, H, 2]
      targets      [B, H, A, 7]  (first 2 dims are x,y)
      targets_mask [B, H, A]     (1 valid, 0 pad)

    Mask semantics:
      - If an agent is invalid: mask is all 0 over H.
      - If valid: mask is 1 for first valid_len steps and 0 afterwards (per agent).
        (You can change to arbitrary patterns if you want later.)

    Pred generation:
      - mode="random_pred": pred is fully random (independent of GT).
      - mode="partial_corrupt": pred starts as a copy of GT, then for up to N valid agents
        we replace a random contiguous segment [T:T+K) with random values (or GT+noise).
    """
    if device is None:
        device = torch.device("cpu")
    if seed is not None:
        # Make this function deterministic (for reproducible samples)
        g = torch.Generator(device=device)
        g.manual_seed(seed)
    else:
        g = None

    assert A == 8 and H == 12 and B == 4, "You requested fixed sizes B=4, A=8, H=12."
    valid_len_max = min(valid_len_max, H)
    seg_len_max = min(seg_len_max, H)

    # -----------------------
    # 1) Build targets_mask: [B,H,A]
    # -----------------------
    targets_mask = torch.zeros((B, H, A), device=device, dtype=torch.long)

    # agent_valid[b,a] in {0,1}
    if g is None:
        agent_valid = (torch.rand((B, A), device=device) < p_agent_valid)
    else:
        agent_valid = (torch.rand((B, A), device=device, generator=g) < p_agent_valid)

    # For each valid agent, choose a valid length in [valid_len_min, valid_len_max]
    valid_lens = torch.zeros((B, A), device=device, dtype=torch.long)
    if valid_len_min < 1:
        raise ValueError("valid_len_min must be >= 1")
    if valid_len_min > valid_len_max:
        raise ValueError("valid_len_min must be <= valid_len_max")

    for b in range(B):
        for a in range(A):
            if agent_valid[b, a]:
                if g is None:
                    L = torch.randint(valid_len_min, valid_len_max + 1, (1,), device=device).item()
                else:
                    L = torch.randint(valid_len_min, valid_len_max + 1, (1,), device=device, generator=g).item()
                valid_lens[b, a] = L
                targets_mask[b, :L, a] = 1  # prefix-valid

    # -----------------------
    # 2) Build GT trajectory: targets[..., :2]
    #    We’ll generate a random walk per agent.
    # -----------------------
    # initial positions: [B,A,1,2]
    if g is None:
        start = torch.randn((B, A, 1, 2), device=device, dtype=dtype) * gt_start_scale
        steps = torch.randn((B, A, H, 2), device=device, dtype=dtype) * gt_step_scale
    else:
        start = torch.randn((B, A, 1, 2), device=device, dtype=dtype, generator=g) * gt_start_scale
        steps = torch.randn((B, A, H, 2), device=device, dtype=dtype, generator=g) * gt_step_scale

    gt_xy = start + torch.cumsum(steps, dim=2)  # [B,A,H,2]

    # targets: [B,H,A,7]
    targets = torch.zeros((B, H, A, 7), device=device, dtype=dtype)
    targets[..., :2] = gt_xy.permute(0, 2, 1, 3)  # [B,H,A,2] fill in x,y

    # Fill the rest dims 2:7 with something (random) just to match shape
    if g is None:
        targets[..., 2:] = torch.randn((B, H, A, 5), device=device, dtype=dtype)
    else:
        targets[..., 2:] = torch.randn((B, H, A, 5), device=device, dtype=dtype, generator=g)

    # -----------------------
    # 3) Build pred: [B,A,H,2]
    # -----------------------
    meta: Dict[str, List] = {"corrupt_agents": [], "corrupt_segments": []}

    if mode == "random_pred":
        if g is None:
            pred = torch.randn((B, A, H, 2), device=device, dtype=dtype) * pred_random_scale
        else:
            pred = torch.randn((B, A, H, 2), device=device, dtype=dtype, generator=g) * pred_random_scale
        # No corruption metadata
        for b in range(B):
            meta["corrupt_agents"].append([])
            meta["corrupt_segments"].append([])

    elif mode == "partial_corrupt":
        pred = gt_xy.clone()  # start identical to GT everywhere

        for b in range(B):
            valid_agents = torch.nonzero(agent_valid[b], as_tuple=False).flatten().tolist()

            # Choose up to n_corrupt_agents from valid agents
            if len(valid_agents) == 0:
                meta["corrupt_agents"].append([])
                meta["corrupt_segments"].append([])
                continue

            n_pick = min(n_corrupt_agents, len(valid_agents))
            # random permutation
            if g is None:
                perm = torch.randperm(len(valid_agents), device=device)
            else:
                perm = torch.randperm(len(valid_agents), device=device, generator=g)

            picked = [valid_agents[i] for i in perm[:n_pick].tolist()]
            segs_this: List[Tuple[int, int]] = []

            for a in picked:
                L = int(valid_lens[b, a].item())
                # pick segment length K, start T within [0, L-1]
                if g is None:
                    K = int(torch.randint(seg_len_min, seg_len_max + 1, (1,), device=device).item())
                else:
                    K = int(torch.randint(seg_len_min, seg_len_max + 1, (1,), device=device, generator=g).item())

                # ensure we only corrupt within valid region
                K = max(1, min(K, L))
                max_T = max(0, L - K)
                if g is None:
                    T = int(torch.randint(0, max_T + 1, (1,), device=device).item())
                else:
                    T = int(torch.randint(0, max_T + 1, (1,), device=device, generator=g).item())

                # corrupt pred[b,a,T:T+K]
                if corrupt_as_noise:
                    if g is None:
                        noise = torch.randn((K, 2), device=device, dtype=dtype) * corrupt_noise_scale
                    else:
                        noise = torch.randn((K, 2), device=device, dtype=dtype, generator=g) * corrupt_noise_scale
                    pred[b, a, T : T + K] = gt_xy[b, a, T : T + K] + noise
                else:
                    if g is None:
                        pred[b, a, T : T + K] = torch.randn((K, 2), device=device, dtype=dtype) * corrupt_random_scale
                    else:
                        pred[b, a, T : T + K] = torch.randn((K, 2), device=device, dtype=dtype, generator=g) * corrupt_random_scale

                segs_this.append((T, T + K))

            meta["corrupt_agents"].append(picked)
            meta["corrupt_segments"].append(segs_this)

    else:
        raise ValueError(f"Unknown mode: {mode}")

    return pred, targets, targets_mask, meta

### Manually Generated Sample Verification

In [48]:
# one valid agent off by one dy for all points

pred = [[1, 14], [1, 15], [1, 16], [1, 17], [1, 18], [1, 19], [1, 20], [1, 21], [1, 22], [1, 23], [1, 24], [1, 25]]
targets = [[1, 15], [1, 16], [1, 17], [1, 18], [1, 19], [1, 20], [1, 21], [1, 22], [1, 23], [1, 24], [1, 25], [1, 26]]
pred = torch.tensor(pred).view(1, 1, 12, 2).float()  # [B,A,H,2]
targets = torch.tensor(targets).view(1, 12, 1, 2).float()  # [B,H,A,2]
targets = torch.cat([targets, torch.zeros((1, 12, 1, 5))], dim=-1)  # [B,H,A,7]
mask = torch.ones((1, 12, 1), dtype=torch.long)  # [B,H,A]
#print("Pred:", pred)
#print("Targets:", targets)

"""
correct:    [B, A] (int {0,1} by default, or bool if return_bool=True)
agent_valid:[B, A] (bool) True iff the agent has at least 1 valid future step
"""
for mode in ["all_points", "mean", "final_point"]:
    correct, agent_valid = agent_traj_correct(
        pred=pred,
        targets=targets,
        targets_mask=mask,
        threshold=1,
        mode=mode,
        metric="l2",
        return_bool=False,
    )  # correct_agent: [B,A] {0,1}, agent_valid: [B,A] bool
    print("-----")
    print("Mode:", mode)
    print("Correct:", correct)
    print("Agent valid:", agent_valid)

-----
Mode: all_points
Correct: tensor([[1]])
Agent valid: tensor([[True]])
-----
Mode: mean
Correct: tensor([[1]])
Agent valid: tensor([[True]])
-----
Mode: final_point
Correct: tensor([[1]])
Agent valid: tensor([[True]])


In [50]:
# one valid agent off by two point for one point and one point for all other points
pred = [[1, 14], [1, 15], [1, 16], [1, 17], [1, 18], [1, 19], [1, 20], [1, 21], [1, 22], [1, 23], [1, 24], [1, 24]]
targets = [[1, 15], [1, 16], [1, 17], [1, 18], [1, 19], [1, 20], [1, 21], [1, 22], [1, 23], [1, 24], [1, 25], [1, 26]]
pred = torch.tensor(pred).view(1, 1, 12, 2).float()  # [B,A,H,2]
targets = torch.tensor(targets).view(1, 12, 1, 2).float()  # [B,H,A,2]
targets = torch.cat([targets, torch.zeros((1, 12, 1, 5))], dim=-1)  # [B,H,A,7]
mask = torch.ones((1, 12, 1), dtype=torch.long)  # [B,H,A]
#print("Pred:", pred)
#print("Targets:", targets)

"""
correct:    [B, A] (int {0,1} by default, or bool if return_bool=True)
agent_valid:[B, A] (bool) True iff the agent has at least 1 valid future step
"""
for mode in ["all_points", "mean", "final_point"]:
    correct, agent_valid = agent_traj_correct(
        pred=pred,
        targets=targets,
        targets_mask=mask,
        threshold=1.2,
        mode=mode,
        metric="l2",
        return_bool=False,
        verbose=False
    )  # correct_agent: [B,A] {0,1}, agent_valid: [B,A] bool
    print("-----")
    print("Mode:", mode)
    print("Correct:", correct)
    print("Agent valid:", agent_valid)

-----
Mode: all_points
Correct: tensor([[0]])
Agent valid: tensor([[True]])
-----
Mode: mean
Correct: tensor([[1]])
Agent valid: tensor([[True]])
-----
Mode: final_point
Correct: tensor([[0]])
Agent valid: tensor([[True]])


In [68]:
# two valid agents, one correct, one incorrect
pred = [[[1, 14], [1, 15], [1, 16], [1, 17], [1, 18], [1, 19], [1, 20], [1, 21], [1, 22], [1, 23], [1, 24], [1, 23]],
        [[2, 14], [2, 15], [2, 16], [2, 17], [2, 18], [2, 19], [2, 20], [2, 21], [2, 22], [2, 23], [2, 24], [2, 25]]]
targets = [[[1, 15], [1, 16], [1, 17], [1, 18], [1, 19], [1, 20], [1, 21], [1, 22], [1, 23], [1, 24], [1, 25], [1, 26]],
           [[2, 15], [2, 16], [2, 17], [2, 18], [2, 19], [2, 20], [2, 21], [2, 22], [2, 23], [2, 24], [2, 25], [2, 26]]]
pred = torch.tensor(pred).view(1, 2, 12, 2).float()  # [B,A,H,2]
targets = torch.tensor(targets).view(1, 2, 12, 2).float()  # [B,A,H,2]
targets = torch.cat([targets, torch.zeros((1, 2, 12, 5))], dim=-1)  # [B,A,H,7]
targets = targets.permute(0, 2, 1, 3).contiguous()  # [B,H,A,7]
mask = torch.ones((1, 12, 2), dtype=torch.long)  # [B,H,A]
print("Pred:", pred.shape)
print(pred[0, 0, :])
print(targets.permute(0, 2, 1, 3)[0, 0, :, :2])
#print("Targets:", targets)
"""
correct:    [B, A] (int {0,1} by default, or bool if return_bool=True)
agent_valid:[B, A] (bool) True iff the agent has at least 1 valid future step
"""
for mode in ["all_points", "mean", "final_point"]:
    correct, agent_valid = agent_traj_correct(
        pred=pred,
        targets=targets,
        targets_mask=mask,
        threshold=1.2,
        mode=mode,
        metric="l2",
        return_bool=False,
        verbose=False
    )  # correct_agent: [B,A] {0,1}, agent_valid: [B,A] bool
    print("-----")
    print("Mode:", mode)
    print("Correct:", correct)
    print("Agent valid:", agent_valid)

Pred: torch.Size([1, 2, 12, 2])
tensor([[ 1., 14.],
        [ 1., 15.],
        [ 1., 16.],
        [ 1., 17.],
        [ 1., 18.],
        [ 1., 19.],
        [ 1., 20.],
        [ 1., 21.],
        [ 1., 22.],
        [ 1., 23.],
        [ 1., 24.],
        [ 1., 23.]])
tensor([[ 1., 15.],
        [ 1., 16.],
        [ 1., 17.],
        [ 1., 18.],
        [ 1., 19.],
        [ 1., 20.],
        [ 1., 21.],
        [ 1., 22.],
        [ 1., 23.],
        [ 1., 24.],
        [ 1., 25.],
        [ 1., 26.]])
-----
Mode: all_points
Correct: tensor([[0, 1]])
Agent valid: tensor([[True, True]])
-----
Mode: mean
Correct: tensor([[1, 1]])
Agent valid: tensor([[True, True]])
-----
Mode: final_point
Correct: tensor([[0, 1]])
Agent valid: tensor([[True, True]])


In [71]:
# one valid agents, one correct, one incorrect (invalid)
pred = [[[1, 14], [1, 15], [1, 16], [1, 17], [1, 18], [1, 19], [1, 20], [1, 21], [1, 22], [1, 23], [1, 24], [1, 24]],
        [[2, 14], [2, 15], [2, 16], [2, 17], [2, 18], [2, 19], [2, 20], [2, 21], [2, 22], [2, 23], [2, 24], [2, -25]]]
targets = [[[1, 15], [1, 16], [1, 17], [1, 18], [1, 19], [1, 20], [1, 21], [1, 22], [1, 23], [1, 24], [1, 25], [1, 25]],
           [[2, 15], [2, 16], [2, 17], [2, 18], [2, 19], [2, 20], [2, 21], [2, 22], [2, 23], [2, 24], [2, 25], [2, 26]]]
pred = torch.tensor(pred).view(1, 2, 12, 2).float()  # [B,A,H,2]
targets = torch.tensor(targets).view(1, 2, 12, 2).float()  # [B,A,H,2]
targets = torch.cat([targets, torch.zeros((1, 2, 12, 5))], dim=-1)  # [B,A,H,7]
targets = targets.permute(0, 2, 1, 3).contiguous()  # [B,H,A,7]
mask = torch.ones((1, 12, 1), dtype=torch.long)  # [B,H,A]
mask = torch.cat([mask, torch.zeros((1, 12, 1), dtype=torch.long)], dim=-1)  # second agent invalid

#print("Targets:", targets)
"""
correct:    [B, A] (int {0,1} by default, or bool if return_bool=True)
agent_valid:[B, A] (bool) True iff the agent has at least 1 valid future step
"""
for mode in ["all_points", "mean", "final_point"]:
    correct, agent_valid = agent_traj_correct(
        pred=pred,
        targets=targets,
        targets_mask=mask,
        threshold=1,
        mode=mode,
        metric="l2",
        return_bool=False,
        verbose=False
    )  # correct_agent: [B,A] {0,1}, agent_valid: [B,A] bool
    print("-----")
    print("Mode:", mode)
    print("Correct:", correct)
    print("Agent valid:", agent_valid)

-----
Mode: all_points
Correct: tensor([[1, 0]])
Agent valid: tensor([[ True, False]])
-----
Mode: mean
Correct: tensor([[1, 0]])
Agent valid: tensor([[ True, False]])
-----
Mode: final_point
Correct: tensor([[1, 0]])
Agent valid: tensor([[ True, False]])


### Automatically generated sample verification

In [72]:
# generat synthetic batch for testing, agent number always fixed to 8, trajectory length always fixed to 12, batch size fixed to 4
pred, targets, mask, meta = generate_synthetic_batch(seed=0, mode="random_pred")

In [75]:
pred.shape

torch.Size([4, 8, 12, 2])

In [73]:
print(pred[0, 0, :, :])
print(targets.permute(0,2,1,3)[0, 0, :, :2]) # B, H, A, 7 -> B, A, H, 2
print(mask.permute(0, 2, 1)[0, 0, :]) # B,H,A -> B, A, H
print(mask.permute(0, 2, 1)[0, :, :])

tensor([[  9.0151, -11.8984],
        [-11.9716,   0.0734],
        [  1.6147, -10.4845],
        [ -4.8083,   2.8534],
        [ 10.5641,  -1.1665],
        [  6.3219,   8.4026],
        [-21.8774,   3.0291],
        [ -9.5619,  -1.5423],
        [  7.7175,   5.2484],
        [ -9.5467, -10.8461],
        [ -5.2579,  -9.6969],
        [-10.5640,  -7.5616]])
tensor([[-3.9850, -6.0326],
        [-4.6243, -5.9861],
        [-4.9574, -5.6821],
        [-5.3224, -6.1238],
        [-4.9926, -5.8855],
        [-5.5007, -5.7953],
        [-5.4466, -6.1727],
        [-5.3244, -6.2114],
        [-5.3694, -5.8465],
        [-6.2920, -5.8590],
        [-5.6073, -4.5305],
        [-5.1148, -4.6603]])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 

In [78]:
""" 
      returns:
      sample_ok:     [B] int {0,1}  -- whether the sample passes the criterion
      num_correct:   [B] long       -- number of valid agents that are correct
      num_valid:     [B] long       -- number of valid agents
      correct_mask:  [B, A] long {0,1} or None  -- per-agent correctness (only if return_counts=True)
"""
output = batch_correct_by_agent_count(pred, targets, mask, threshold=20, mode="mean", metric="l2")
print(output)
print(output[0].shape, output[0])
print(output[1].shape, output[1])

(tensor([1, 0, 0, 0]), tensor([6, 6, 4, 4]), tensor([6, 8, 5, 6]), tensor([[1, 0, 1, 1, 1, 1, 1, 0],
        [0, 1, 1, 1, 0, 1, 1, 1],
        [0, 0, 1, 1, 1, 0, 1, 0],
        [0, 1, 0, 0, 1, 1, 1, 0]]))
torch.Size([4]) tensor([1, 0, 0, 0])
torch.Size([4]) tensor([6, 6, 4, 4])


In [19]:
pred, targets, mask, meta = generate_synthetic_batch(seed=0, mode="partial_corrupt", n_corrupt_agents=3)

In [20]:
print(pred[0, 0, :, :])
print(targets.permute(0,2,1,3)[0, 0, :, :2]) # B, H, A, 7 -> B, A, H, 2
print(mask.permute(0, 2, 1)[0, 0, :]) # B,H,A -> B, A, H


tensor([[ -3.9850,  -6.0326],
        [ -4.6243,  -5.9861],
        [ -4.9574,  -5.6821],
        [ -5.3224,  -6.1238],
        [ -4.9926,  -5.8855],
        [ 16.1315,  -3.9227],
        [  5.0291,  -8.2984],
        [-14.3538, -10.1898],
        [ -3.1441,  18.9054],
        [ -3.1398,   3.3174],
        [ -5.6073,  -4.5305],
        [ -5.1148,  -4.6603]])
tensor([[-3.9850, -6.0326],
        [-4.6243, -5.9861],
        [-4.9574, -5.6821],
        [-5.3224, -6.1238],
        [-4.9926, -5.8855],
        [-5.5007, -5.7953],
        [-5.4466, -6.1727],
        [-5.3244, -6.2114],
        [-5.3694, -5.8465],
        [-6.2920, -5.8590],
        [-5.6073, -4.5305],
        [-5.1148, -4.6603]])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])


In [21]:
""" 
      returns:
      sample_ok:     [B] int {0,1}  -- whether the sample passes the criterion
      num_correct:   [B] long       -- number of valid agents that are correct
      num_valid:     [B] long       -- number of valid agents
      correct_mask:  [B, A] long {0,1} or None  -- per-agent correctness (only if return_counts=True)
"""
output = batch_correct_by_agent_count(pred, targets, mask, threshold=20, mode="all_points", metric="l2")
print(output)

(tensor([0, 0, 0, 0]), tensor([3, 7, 2, 3]), tensor([6, 8, 5, 6]), tensor([[0, 0, 0, 1, 1, 0, 1, 0],
        [1, 1, 1, 1, 1, 1, 0, 1],
        [1, 0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 0, 0]]))
